# MD1 — Pit Stop Duration: Regression Pipeline

Predict total pit lane time (entry → exit) per stop using only information available before the car enters the pit lane. This notebook contains temporal split, baselines, an optional historical feature (shift(1)), model pipelines, evaluation, and the required failure analysis.

In [65]:
# Imports and constants
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')
RANDOM_SEED = 414

# Load data (CSV provided)
df = pd.read_csv('md1_pitstops_2019_2024.csv')
print('Loaded', df.shape, 'rows')
df.head()

Loaded (4170, 10) rows


,season,round,circuit,Driver,Team,LapNumber,Stint,Compound,TyreLife,pit_stop_duration
0,2019,1,Australian Grand Prix,GAS,Red Bull Racing,37.0,1.0,MEDIUM,37.0,20.333
1,2019,1,Australian Grand Prix,PER,Aston Martin,13.0,1.0,SOFT,16.0,20.726
2,2019,1,Australian Grand Prix,LEC,Ferrari,28.0,1.0,SOFT,31.0,21.765
3,2019,1,Australian Grand Prix,STR,Aston Martin,27.0,1.0,MEDIUM,27.0,22.339
4,2019,1,Australian Grand Prix,MAG,Haas F1 Team,14.0,1.0,SOFT,17.0,22.482


In [66]:
# Quick target summary
print(df['pit_stop_duration'].describe())

count    4170.000000
mean       24.888535
std         4.852716
min        15.016000
25%        21.990750
50%        23.665500
75%        26.771750
max        59.099000
Name: pit_stop_duration, dtype: float64


**Leakage check — For each feature: would we know it at pit entry?**

Feature | Safe? | Reasoning
---|---|---
season | ✅ | Known before the race
round | ✅ | Known before the race
circuit | ✅ | Track known before race
Driver | ✅ | Driver known
Team | ✅ | Constructor known
LapNumber | ✅ | Current lap counter visible
Stint | ✅ | Stop count known (which stop this ends)
Compound | ✅ | Tyre in use visible before entry
TyreLife | ✅ | Laps on current tyre known
historical_team_avg_duration (shift(1)) | ✅ | Uses only prior races
next_compound | ❌ | Decided during stop, not before

## 1. Temporal split

In [67]:
# Temporal split: train seasons <= 2022, test seasons >= 2023
train = df[df['season'] <= 2022].copy()
test = df[df['season'] >= 2023].copy()
print('Train rows:', len(train), 'Test rows:', len(test))

Train rows: 2593 Test rows: 1577


## 2. Baselines (required)

In [68]:
# Baselines: naive mean and team mean (trained on train set)
from sklearn.metrics import mean_absolute_error
y_train = train['pit_stop_duration']
y_test = test['pit_stop_duration']

# Naive mean baseline
naive_pred = np.repeat(y_train.mean(), len(y_test))
naive_mae = mean_absolute_error(y_test, naive_pred)

# Team mean baseline: compute mean per Team in training set and map to test
team_mean = y_train.groupby(train['Team']).mean().to_dict()
team_pred = test['Team'].map(team_mean).fillna(y_train.mean()).values
team_mae = mean_absolute_error(y_test, team_pred)

print('Naive mean MAE:', round(naive_mae, 3))
print('Team mean MAE :', round(team_mae, 3))

results = []
results.append({'model':'Naive mean','mae':naive_mae})
results.append({'model':'Team mean','mae':team_mae})

Naive mean MAE: 3.454
Team mean MAE : 3.439


## 3. Historical features (optional but encouraged)

In [69]:
# Historical team-level feature using prior races only (shift(1) at race granularity)
# Step 1: compute team x race mean
df['race_id'] = df['season'].astype(str) + '_' + df['round'].astype(str)
team_race_mean = df.groupby(['Team','season','round'])['pit_stop_duration'].mean().reset_index().rename(columns={'pit_stop_duration':'team_race_mean'})
# sort by team then season/round and compute cumulative mean shifted by 1 (prior races only)
team_race_mean = team_race_mean.sort_values(['Team','season','round'])
team_race_mean['historical_team_avg_duration'] = team_race_mean.groupby('Team')['team_race_mean'].expanding().mean().reset_index(level=0, drop=True).groupby(team_race_mean['Team']).shift(1)
team_race_mean = team_race_mean.drop(columns=['team_race_mean'])
# merge back to main df (per race)
df = df.merge(team_race_mean, on=['Team','season','round'], how='left')

# Now recompute train/test views (they inherit the new column)
train = df[df['season'] <= 2022].copy()
test = df[df['season'] >= 2023].copy()
# Fill historical_team_avg_duration in train with prior races; for earliest races it will be NaN — we can impute with train mean later in pipeline
train['historical_team_avg_duration'].isna().sum(), test['historical_team_avg_duration'].isna().sum()

(22, 0)

## 4. Model

In [70]:
# Features to use (all temporally safe)
features = ['circuit', 'Team', 'TyreLife', 'LapNumber', 'Stint', 'Compound', 'historical_team_avg_duration']
X_train = train[features].copy()
X_test = test[features].copy()
y_train = train['pit_stop_duration']
y_test = test['pit_stop_duration']

# Column types
cat_feats = ['circuit', 'Team', 'Compound']
num_feats = ['TyreLife', 'LapNumber', 'Stint', 'historical_team_avg_duration']

# Preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='mean'), num_feats),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), cat_feats)
    ])

In [71]:
# Regression Model Pipeline (Ridge and Random Forest)

# Ridge pipeline
ridge_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('scaler', StandardScaler(with_mean=False)),
    ('model', Ridge(alpha=40.0, random_state=RANDOM_SEED))
])

# Fit and predict for Ridge
ridge_pipe.fit(X_train, y_train)
ridge_preds = ridge_pipe.predict(X_test)
ridge_mae = mean_absolute_error(y_test, ridge_preds)

print('Ridge MAE:', round(ridge_mae, 3))


# Random Forest pipeline
rf_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=100, max_depth=10, random_state=RANDOM_SEED, n_jobs=-1))
])

# Fit and predict for Random Forest
rf_pipe.fit(X_train, y_train)
rf_preds = rf_pipe.predict(X_test)
rf_mae = mean_absolute_error(y_test, rf_preds)

print('Random Forest MAE:', round(rf_mae, 3))

Ridge MAE: 2.39
Random Forest MAE: 2.562


## 5. Comparison table

In [72]:
# Comparison Table
results = [
    {'Model': 'Naive mean', 'MAE': naive_mae},
    {'Model': 'Team mean', 'MAE': team_mae},
    {'Model': 'Ridge Regression', 'MAE': ridge_mae},
    {'Model': 'Random Forest', 'MAE': rf_mae}
]

comparison_df = pd.DataFrame(results).sort_values(by='MAE').reset_index(drop=True)
print("=== Validation Results ===")
display(comparison_df)

=== Validation Results ===


,Model,MAE
0,Ridge Regression,2.389926
1,Random Forest,2.562050
2,Team mean,3.439362
3,Naive mean,3.453893


## Why Our Model Fails
One specific area where our model consistently struggles is predicting pit stop durations during Safety Car or Virtual Safety Car (VSC) periods. Because the race pace is significantly reduced under these conditions, the relative time lost transiting the pit lane changes, and teams may execute double-stacked pit stops which increase wait times unpredictably for the second driver. 

My hypothesis for why it fails is that our model currently lacks any feature indicating the race status (green flag vs. SC/VSC) or the proximity of the driver's teammate on track. Without knowing if a Safety Car was deployed or if traffic was congested in the pit lane just prior to entry, the model assumes standard green-flag racing conditions and underestimates or overestimates the transit and box phases.

If I had 30 more minutes, I would engineer a `is_safety_car` boolean feature (derived from race control messages just prior to the stop) and a `teammate_in_pits_recently` feature indicating if the driver's teammate pitted on the same or preceding lap.